## Workflow Documentation

## Abstract

In this project I explored watershed delineation methods in ArcGIS, putting together many different geoprocessing tools on raster data. After initial python scripting on an arbitrary pour point, I imported EPA Region 9 Superfund boundaries and analyzed watershed intersections with the three contiguous superfund sites in the San Fernando Valley. First I generated a weighted polygon layer for my pour point's watershed to look pretty and to visually represent superfund site contribution to the watershed. This included intermediate rasters such as scrubbed DEM, flow direction and accumulation, a conditional stream layer that filters out streams with less than 5,000 contribution cells, and clipped versions of datasets to reduce geoprocessing to only my region of interest.

For the heavier analysis, I generated superfund site inlets/outlets using Deterministic 8-neighbor methods on the superfund boundary using my previously generated flow direction dataset and an intermediate boolean dataset to store raster cells that are inside the boundary. After generating inlet/outlet cells, I plotted them on the map as point layers and summed accumulation at each point.

Finally, I calculated accumulation contribution from inside the superfund boundary to the LA River at my pour point by subtracting superfund inlets from superfund outlets (2.6% of the accumulation at the pour point). There are many ways to slice this data; subtracting off inlets was primarily a choice to extend analysis for my own experience.

Implications of this analysis include support for prioritizing cleanup funding, identifying potential sample locations, and piecing together water permitting for facilities in this watershed. Of course, the methods I developed here are more broadly useful for other areas, as I can now easily apply the methods to any pour point and analyze against any area, not just this portion of the LA river and this particular superfund site.

### Data Gathering
The purpose of this project was to get familiar with raster datasets, and oh boy if I thought geoprocessing on polygons took a long time, I ended up in a whole new world with this one.

Watershed delineation is a classic raster dataset practice, and despite my experience in environmental engineering I've not touched on it during my career (I work mostly in air). I started by grabbing DEM data from USGS here: https://apps.nationalmap.gov/downloader/

There's a lot of data here, so I first did some data manipulation in Excel to ID the most recent DEMs and prepare them for download with a Python loop. I made the mistake of thinking I could reasonably download all of California's DEMs (and I did); however, it's about 40 GB and I ended up removing most of the files after identifying a region of interest.

Later in the project I pulled EPA Region 9 Superfund boundaries here: https://www.arcgis.com/home/item.html?id=7ff01522a7424616a8b9faa03ec4b0ce#overview

I then filtered this down for analysis to the three contiguous site boundaries in the San Fernando Valley.

### Data Scrubbing
I put a lot of effort into data scrubbing here. In Data Gathering I mentioned filtering down to my region of interest for both datasets; that was a good start.

Next I created a new raster from all my DEM TIF files, replaced zeros with null values, reprojected the dataset in meters for analysis, and filled it.

Typical steps followed; create flow accumulation, flow direction rasters. Create a pour point, snap it to the raster. Calculate a watershed raster.

Next I created a watershed polygon from the raster and dissolved it. After creating a filtered streams dataset from my flow accumulation (anything over 5,000 contribution cells), I clipped my flow direction and streams to my region of interest, streamlinked the streams and built a table with their accumulation, and created a stream polygon layer. Finally, I joined the table to the layer and symbolized streams based on the accumulation that I added.

One thing that proved to be a huge pain was zeroes in the streams dataset I generated. After stream linking, creating the feature class using streamtofeature was treating zeroes as data, resulting in a HUGE processing time and a basically bypassing the flow accumulation filter I had applied. After a few crashes, I figured it out and setnull on the streams dataset to fix the issue.

### Analysis
Now for something meaningful, let's run some geoprocessing! The obvious question is "how does the superfund site contribute to the watershed?"

I was particularly interested in building out some automation, and liked the D8 protocol that the flow direction was using. So. Of course I had to use that to calculate points where streams entered and exited the superfund boundary. Turns out it's a little complicated. There was surprisingly a lot of debugging related to how to save files for this, made all the worse by the fact that I had copied the boundary layer to my gdb, and thus the geoprocessing was trying to run on the entire layer (2,000+ sites), not the three that I had identified using a definition query on the web hosted layer.

So after preceding the code with a select on those three sites, AI gave me some options for fixing the file saving issue. The solution it identified was to export temporary files as TIFs and then to import the final file into my gdb. Smart.

To accomplish my goal, I created two new rasters; one identified whether a cell is within the site boundary, easy. The second, harder, identifies whether the cell downstream of each cell is within the site boundary. I did this by looping over every D8 direction for a cell, checking which one was true, and then asking if a cell shift in that direction is within the site boundary.

Next I took the two new rasters and said hey if this cell is inside and it's downstream cell is outside, that's an outlet. And vice versa. And put coordinates on them.

The rest was easy; I defined a function to sum accumulation over a set of points and passed the ins/outs to the function, took the total out minus the total in, and divided it by the total accumulation at my pour point to get a percentage of flow originating inside the superfund site boundary.